# PulmoVision - GPU training on Kaggle

Add these inputs with **Add Data**: `tawsifurrahman/covid19-radiography-database`, `tawsifurrahman/tuberculosis-tb-chest-xray-dataset`, `mohamedhanyyy/chest-ctscan-images`. Set **Accelerator: GPU** and **Run All**. Download `/kaggle/working/models` and `/kaggle/working/experiments` and copy them into the project's `models/` and `experiments/` folders.

In [ ]:
!pip install -q segmentation-models-pytorch==0.3.4 timm==0.9.7 pydicom==3.0.1

In [ ]:
import glob, os
def find(pattern):
    hits = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    assert hits, f'dataset folder {pattern} not found - add it with Add Data'
    return hits[0]
os.environ['COVID_ROOT'] = find('COVID-19_Radiography_Dataset')
os.environ['TB_ROOT'] = find('TB_Chest_Radiography_Database')
os.environ['CT_ROOT'] = os.path.dirname(find('Data/train'))
print({k: os.environ[k] for k in ('COVID_ROOT', 'TB_ROOT', 'CT_ROOT')})

In [ ]:
!mkdir -p work/backend/app/services work/ml

In [ ]:
!mkdir -p work/backend/app && touch work/backend/app/__init__.py

In [ ]:
!mkdir -p work/backend/app/services && touch work/backend/app/services/__init__.py

In [ ]:
%%writefile work/backend/app/services/imaging.py
"""Image loading, preprocessing and overlay helpers shared by the API and the training scripts."""
import io

import cv2
import numpy as np
from PIL import Image

CLS_SIZE = 224  # classifier input
SEG_SIZE = 256  # U-Net input
VIEW_SIZE = 512  # images shown in the UI
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class ImageError(ValueError):
    pass


def load_image(data: bytes, filename: str = ""):
    """Return (gray uint8, rgb uint8, meta). Accepts PNG / JPG / DICOM."""
    name = filename.lower()
    is_dicom = name.endswith((".dcm", ".dicom")) or data[128:132] == b"DICM"
    if is_dicom:
        import pydicom
        from pydicom.pixels import apply_voi_lut

        try:
            ds = pydicom.dcmread(io.BytesIO(data), force=True)
            arr = ds.pixel_array.astype(np.float32)
        except Exception as e:  # noqa: BLE001
            raise ImageError(f"Could not read the DICOM file: {e}")
        if arr.ndim == 3 and arr.shape[-1] != 3:  # multi-frame: take the middle slice
            arr = arr[arr.shape[0] // 2]
        try:
            arr = apply_voi_lut(arr, ds).astype(np.float32)
        except Exception:  # noqa: BLE001
            arr = arr * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))
        if arr.ndim == 3:
            arr = arr.mean(axis=-1)
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            arr = arr.max() - arr
        arr = (arr - arr.min()) / max(float(arr.max() - arr.min()), 1e-6) * 255
        gray = arr.astype(np.uint8)
        rgb = np.stack([gray] * 3, axis=-1)
        meta = {"format": "DICOM", "dicom_modality": str(getattr(ds, "Modality", ""))}
    else:
        try:
            img = Image.open(io.BytesIO(data))
            img.load()
        except Exception:  # noqa: BLE001
            raise ImageError("The file is not a readable PNG, JPG or DICOM image")
        if img.format not in ("PNG", "JPEG", "MPO"):
            raise ImageError(f"Unsupported image format {img.format}; use PNG, JPG or DICOM")
        rgb = np.array(img.convert("RGB"))
        gray = np.array(img.convert("L"))
        meta = {"format": "JPEG" if img.format == "MPO" else img.format, "dicom_modality": ""}
    if min(gray.shape) < 64:
        raise ImageError("Image is too small (minimum 64 px per side)")
    meta["width"], meta["height"] = int(gray.shape[1]), int(gray.shape[0])
    return gray, rgb, meta


def colourfulness(rgb: np.ndarray) -> float:
    """Mean HSV saturation (0-1) of reasonably bright pixels; radiographs are close to 0."""
    small = cv2.resize(rgb, (128, 128), interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(small, cv2.COLOR_RGB2HSV)
    bright = hsv[..., 2] > 40
    if bright.sum() < 50:
        return 0.0
    return float(hsv[..., 1][bright].mean() / 255.0)


def preprocess(gray: np.ndarray, size: int = CLS_SIZE):
    """Resize -> denoise -> CLAHE. Returns the three intermediate uint8 images."""
    resized = cv2.resize(gray, (size, size), interpolation=cv2.INTER_AREA)
    denoised = cv2.fastNlMeansDenoising(resized, None, h=5, templateWindowSize=7, searchWindowSize=15)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(denoised)
    return {"resized": resized, "denoised": denoised, "enhanced": clahe}


def to_tensor_array(enhanced: np.ndarray) -> np.ndarray:
    """uint8 HxW -> float32 3xHxW normalised with ImageNet statistics."""
    x = enhanced.astype(np.float32) / 255.0
    x = np.stack([x, x, x], axis=0)
    return (x - MEAN[:, None, None]) / STD[:, None, None]


def to_seg_array(gray: np.ndarray) -> np.ndarray:
    """U-Net input: CLAHE-enhanced 256x256, scaled 0-1, shape 1xHxW."""
    enh = preprocess(gray, SEG_SIZE)["enhanced"]
    return (enh.astype(np.float32) / 255.0)[None]


def ct_lung_mask(gray: np.ndarray) -> np.ndarray:
    """Classical CT lung field estimate: dark regions enclosed by the body outline."""
    g = cv2.GaussianBlur(gray, (5, 5), 0)
    _, body = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(body)
    if n <= 1:
        return np.ones_like(gray, dtype=np.uint8)
    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    body = (labels == largest).astype(np.uint8) * 255
    contours, _ = cv2.findContours(body, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(body)
    cv2.drawContours(filled, contours, -1, 255, thickness=cv2.FILLED)
    lungs = ((filled > 0) & (body == 0)).astype(np.uint8)
    lungs = cv2.morphologyEx(lungs, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    n, labels, stats, _ = cv2.connectedComponentsWithStats(lungs)
    keep = np.zeros_like(lungs)
    min_area = 0.005 * gray.size
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            keep[labels == i] = 1
    if keep.sum() < 0.02 * gray.size:
        return (filled > 0).astype(np.uint8)  # fall back to the whole body region
    return keep


def png_bytes(img: np.ndarray) -> bytes:
    ok, buf = cv2.imencode(".png", cv2.cvtColor(img, cv2.COLOR_RGB2BGR) if img.ndim == 3 else img)
    return buf.tobytes()


def view(img: np.ndarray) -> np.ndarray:
    return cv2.resize(img, (VIEW_SIZE, VIEW_SIZE), interpolation=cv2.INTER_LINEAR)


def mask_overlay(gray: np.ndarray, mask: np.ndarray) -> np.ndarray:
    base = cv2.cvtColor(view(gray), cv2.COLOR_GRAY2RGB).astype(np.float32)
    m = cv2.resize(mask.astype(np.uint8), (VIEW_SIZE, VIEW_SIZE), interpolation=cv2.INTER_NEAREST) > 0
    tint = np.array([34, 197, 94], dtype=np.float32)
    base[m] = base[m] * 0.6 + tint * 0.4
    out = base.clip(0, 255).astype(np.uint8)
    contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, (22, 163, 74), 2)
    return out


def heatmap_overlay(gray: np.ndarray, cam: np.ndarray) -> np.ndarray:
    base = cv2.cvtColor(view(gray), cv2.COLOR_GRAY2RGB).astype(np.float32)
    c = cv2.resize(cam.astype(np.float32), (VIEW_SIZE, VIEW_SIZE))
    colour = cv2.cvtColor(cv2.applyColorMap((c * 255).astype(np.uint8), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    alpha = (0.25 + 0.35 * c)[..., None]
    return (base * (1 - alpha) + colour * alpha).clip(0, 255).astype(np.uint8)


In [ ]:
%%writefile work/backend/app/services/nets.py
"""Network definitions and class lists (shared by training and inference)."""
import segmentation_models_pytorch as smp
import timm

XRAY_CLASSES = ["COVID-19", "Viral Pneumonia", "Lung Opacity", "Tuberculosis", "Normal"]
CT_CLASSES = ["Adenocarcinoma", "Large-cell Carcinoma", "Squamous-cell Carcinoma", "Normal"]


def make_classifier(num_classes: int, pretrained: bool = False):
    return timm.create_model("efficientnet_b0", pretrained=pretrained, num_classes=num_classes)


def make_backbone(pretrained: bool = False):
    """ImageNet EfficientNet-B0 without its head: 1280-d embeddings for the modality check."""
    return timm.create_model("efficientnet_b0", pretrained=pretrained, num_classes=0)


def make_unet(pretrained: bool = False):
    return smp.Unet("mobilenet_v2", encoder_weights="imagenet" if pretrained else None, in_channels=1, classes=1)


In [ ]:
%%writefile work/ml/common.py
"""Shared helpers for the ml/ scripts: paths, dataset roots, splits and cached arrays."""
import json
import os
import pathlib
import sys

ROOT = pathlib.Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "backend"))

from app.services import imaging  # noqa: E402
from app.services.nets import CT_CLASSES, XRAY_CLASSES  # noqa: E402

SEED = 42
CACHE = ROOT / "data" / "raw" / "cache"
SPLITS = ROOT / "experiments" / "splits.json"
MODELS = ROOT / "models"
EXPERIMENTS = ROOT / "experiments"


def dataset_roots():
    """Locations of the three datasets. Override with env vars (used on Kaggle)."""
    paths_file = ROOT / "data" / "raw" / "paths.json"
    found = json.loads(paths_file.read_text()) if paths_file.exists() else {}
    covid = os.getenv("COVID_ROOT") or (found.get("covid19_radiography", "") and os.path.join(found["covid19_radiography"], "COVID-19_Radiography_Dataset"))
    tb = os.getenv("TB_ROOT") or (found.get("tb_chest_xray", "") and os.path.join(found["tb_chest_xray"], "TB_Chest_Radiography_Database"))
    ct = os.getenv("CT_ROOT") or str(ROOT / "data" / "chest_ct")
    return {"covid": covid, "tb": tb, "ct": ct}


def read_gray(root_key, rel):
    path = os.path.join(dataset_roots()[root_key], rel)
    with open(path, "rb") as f:
        gray, _, _ = imaging.load_image(f.read(), path)
    return gray


In [ ]:
%%writefile work/ml/prepare_data.py
"""Build seeded train/val/test splits and cache preprocessed arrays.

python ml/prepare_data.py            (needs scripts/download_data.py to have run)
Outputs: experiments/splits.json, data/raw/cache/*.npz, data/sample/*
"""
import hashlib
import json
import os
import random
import shutil
from multiprocessing import Pool

import cv2
import numpy as np

from common import CACHE, CT_CLASSES, EXPERIMENTS, ROOT, SEED, SPLITS, XRAY_CLASSES, dataset_roots, imaging, read_gray

# images per class: train / val / test (small, CPU-friendly config)
XRAY_PER_CLASS = {"train": 500, "val": 100, "test": 200}
TB_PER_CLASS = {"train": 400, "val": 100, "test": 200}
SEG_PER_CLASS = {"train": 300, "val": 50, "test": 100}
COVID_DIRS = {"COVID-19": "COVID", "Viral Pneumonia": "Viral Pneumonia", "Lung Opacity": "Lung_Opacity", "Normal": "Normal"}
CT_DIRS = {"adenocarcinoma": 0, "large.cell.carcinoma": 1, "squamous.cell.carcinoma": 2, "normal": 3}


def take(files, counts, rng):
    files = sorted(files)
    rng.shuffle(files)
    out, i = {}, 0
    for split, n in counts.items():
        out[split] = files[i:i + n]
        i += n
    return out


def build_splits():
    rng = random.Random(SEED)
    roots = dataset_roots()
    xray = {s: [] for s in XRAY_PER_CLASS}
    seg = {s: [] for s in SEG_PER_CLASS}
    for label, d in COVID_DIRS.items():
        files = [f"{d}/images/{f}" for f in os.listdir(os.path.join(roots["covid"], d, "images"))]
        counts = XRAY_PER_CLASS if label != "Normal" else {k: v // 2 for k, v in XRAY_PER_CLASS.items()}
        chosen = take(files, counts, rng)
        used = set(sum(chosen.values(), []))
        for s, fs in chosen.items():
            xray[s] += [["covid", f, XRAY_CLASSES.index(label)] for f in fs]
        rest = [f for f in files if f not in used]
        for s, fs in take(rest, SEG_PER_CLASS, rng).items():
            seg[s] += [["covid", f, f.replace("/images/", "/masks/")] for f in fs]
    for label, d, counts in [("Tuberculosis", "Tuberculosis", TB_PER_CLASS),
                             ("Normal", "Normal", {k: v // 2 for k, v in XRAY_PER_CLASS.items()})]:
        files = [f"{d}/{f}" for f in os.listdir(os.path.join(roots["tb"], d))]
        for s, fs in take(files, counts, rng).items():
            xray[s] += [["tb", f, XRAY_CLASSES.index(label)] for f in fs]

    # CT: keep the published split, drop byte-identical duplicates within and across splits
    ct = {"train": [], "val": [], "test": []}
    seen, dropped = set(), 0
    for split, folder in [("train", "train"), ("val", "valid"), ("test", "test")]:
        base = os.path.join(roots["ct"], folder)
        for d in sorted(os.listdir(base)):
            label = next(v for k, v in CT_DIRS.items() if d.startswith(k))
            for f in sorted(os.listdir(os.path.join(base, d))):
                digest = hashlib.md5(open(os.path.join(base, d, f), "rb").read()).hexdigest()
                if digest in seen:
                    dropped += 1
                    continue
                seen.add(digest)
                ct[split].append(["ct", f"{folder}/{d}/{f}", label])
    splits = {"seed": SEED, "xray": xray, "seg": seg, "ct": ct, "ct_duplicates_dropped": dropped}
    SPLITS.parent.mkdir(parents=True, exist_ok=True)
    SPLITS.write_text(json.dumps(splits))
    print({k: {s: len(v) for s, v in splits[k].items()} for k in ("xray", "seg", "ct")}, "CT duplicates dropped:", dropped)
    return splits


def _cls_item(item):
    key, rel, label = item
    return imaging.preprocess(read_gray(key, rel))["enhanced"], label


def _seg_item(item):
    key, rel, mask_rel = item
    gray = read_gray(key, rel)
    mask = cv2.imread(os.path.join(dataset_roots()[key], mask_rel), cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (imaging.SEG_SIZE, imaging.SEG_SIZE), interpolation=cv2.INTER_NEAREST) > 127
    return imaging.to_seg_array(gray)[0], mask


def cache(splits):
    CACHE.mkdir(parents=True, exist_ok=True)
    with Pool(max(1, (os.cpu_count() or 2) - 2)) as pool:
        for task in ("xray", "ct"):
            for split, items in splits[task].items():
                out = pool.map(_cls_item, items, chunksize=16)
                np.savez_compressed(CACHE / f"{task}_{split}.npz", x=np.stack([o[0] for o in out]), y=np.array([o[1] for o in out]))
                print("cached", task, split, len(out))
        for split, items in splits["seg"].items():
            out = pool.map(_seg_item, items, chunksize=16)
            np.savez_compressed(CACHE / f"seg_{split}.npz", x=np.stack([o[0] for o in out]).astype(np.float16), m=np.stack([o[1] for o in out]))
            print("cached seg", split, len(out))


def samples(splits):
    """Copy a small labelled sample of the X-ray test split into data/sample/ (committed)."""
    roots = dataset_roots()
    out = ROOT / "data" / "sample"
    per_class = {}
    for key, rel, label in splits["xray"]["test"]:
        name = XRAY_CLASSES[label]
        if per_class.get(name, 0) >= 6:
            continue
        per_class[name] = per_class.get(name, 0) + 1
        dst = out / "xray" / name.replace(" ", "_").replace("-", "")
        dst.mkdir(parents=True, exist_ok=True)
        shutil.copy(os.path.join(roots[key], rel), dst / os.path.basename(rel))
    for key, rel, mask_rel in splits["seg"]["test"][:6]:
        (out / "lung_masks" / "images").mkdir(parents=True, exist_ok=True)
        (out / "lung_masks" / "masks").mkdir(parents=True, exist_ok=True)
        shutil.copy(os.path.join(roots[key], rel), out / "lung_masks" / "images" / os.path.basename(rel))
        shutil.copy(os.path.join(roots[key], mask_rel), out / "lung_masks" / "masks" / os.path.basename(rel))
    # non-chest images for the modality check (scikit-learn's bundled public-domain photos)
    from sklearn.datasets import load_sample_images

    (out / "other").mkdir(parents=True, exist_ok=True)
    for i, img in enumerate(load_sample_images().images):
        cv2.imwrite(str(out / "other" / f"photo_{i + 1}.jpg"), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    print("samples written to", out)


if __name__ == "__main__":
    s = build_splits()
    cache(s)
    samples(s)
    EXPERIMENTS.mkdir(exist_ok=True)


In [ ]:
%%writefile work/ml/train.py
"""Train PulmoVision models from the cached arrays (run ml/prepare_data.py first).

python ml/train.py xray      EfficientNet-B0, 5 X-ray classes
python ml/train.py ct        EfficientNet-B0, 4 CT classes
python ml/train.py unet      U-Net (MobileNetV2 encoder) lung segmentation
python ml/train.py modality  ImageNet EfficientNet-B0 embedding bank for the X-ray / CT / other check
Options: --epochs N  --device cuda
"""
import argparse
import json
import math
import os
import random
import time

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score

from common import CACHE, CT_CLASSES, EXPERIMENTS, MODELS, SEED, XRAY_CLASSES, imaging
from app.services.nets import make_backbone, make_classifier, make_unet

torch.set_num_threads(os.cpu_count() or 4)


def seed_all():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)


def load(name):
    return np.load(CACHE / f"{name}.npz")


def augment(img, rng):
    h, w = img.shape
    angle = rng.uniform(-8, 8)
    scale = rng.uniform(0.9, 1.1)
    m = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    m[:, 2] += rng.uniform(-0.05, 0.05, 2) * w
    out = cv2.warpAffine(img, m, (w, h), borderMode=cv2.BORDER_REFLECT)
    alpha, beta = rng.uniform(0.85, 1.15), rng.uniform(-15, 15)
    return np.clip(out.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)


def batches(x, y, bs, train, rng):
    idx = np.arange(len(x))
    if train:
        rng.shuffle(idx)
    for i in range(0, len(idx), bs):
        b = idx[i:i + bs]
        imgs = [augment(x[j], rng) if train else x[j] for j in b]
        yield torch.from_numpy(np.stack([imaging.to_tensor_array(im) for im in imgs])), torch.from_numpy(y[b]).long()


@torch.no_grad()
def predict_logits(model, x, device, bs=64):
    model.eval()
    out = []
    for i in range(0, len(x), bs):
        xb = torch.from_numpy(np.stack([imaging.to_tensor_array(im) for im in x[i:i + bs]])).to(device)
        out.append(model(xb).float().cpu())
    return torch.cat(out)


def fit_temperature(logits, y):
    """Temperature scaling on the validation set (calibrated confidence)."""
    t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([t], lr=0.1, max_iter=100)
    yt = torch.from_numpy(y).long()

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / t.exp(), yt)
        loss.backward()
        return loss

    opt.step(closure)
    return float(t.exp().item())


def train_classifier(task, epochs, device):
    classes = XRAY_CLASSES if task == "xray" else CT_CLASSES
    tr, va = load(f"{task}_train"), load(f"{task}_val")
    xtr, ytr, xva, yva = tr["x"], tr["y"], va["x"], va["y"]
    model = make_classifier(len(classes), pretrained=True).to(device)
    for p in list(model.conv_stem.parameters()) + list(model.bn1.parameters()) + list(model.blocks[:2].parameters()):
        p.requires_grad = False  # freeze the earliest layers: faster CPU training, little accuracy cost
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=6e-4, weight_decay=1e-4)
    bs = 32
    steps = epochs * math.ceil(len(xtr) / bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=6e-4, total_steps=steps, pct_start=0.15)
    counts = np.bincount(ytr, minlength=len(classes))
    weights = torch.tensor(len(ytr) / (len(classes) * counts), dtype=torch.float32, device=device)
    rng = np.random.default_rng(SEED)
    best, best_state, history = -1, None, []
    for ep in range(epochs):
        model.train()
        t0, total, n = time.time(), 0.0, 0
        for xb, yb in batches(xtr, ytr, bs, True, rng):
            xb, yb = xb.to(device), yb.to(device)
            loss = F.cross_entropy(model(xb), yb, weight=weights, label_smoothing=0.1)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            total += loss.item() * len(yb)
            n += len(yb)
        logits = predict_logits(model, xva, device)
        pred = logits.argmax(1).numpy()
        acc = float((pred == yva).mean())
        f1 = float(f1_score(yva, pred, average="macro"))
        history.append({"epoch": ep + 1, "loss": total / n, "val_acc": acc, "val_macro_f1": f1, "seconds": round(time.time() - t0, 1)})
        print(history[-1], flush=True)
        if f1 > best:
            best, best_state = f1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    temperature = fit_temperature(predict_logits(model, xva, device), yva)
    MODELS.mkdir(exist_ok=True)
    torch.save({"state_dict": best_state, "classes": classes, "temperature": temperature,
                "arch": "efficientnet_b0", "input_size": imaging.CLS_SIZE}, MODELS / f"{task}_classifier.pt")
    (EXPERIMENTS / f"{task}_training.json").write_text(json.dumps({"history": history, "best_val_macro_f1": best, "temperature": temperature,
                                                                  "train_images": int(len(xtr)), "val_images": int(len(xva))}, indent=2))
    print(f"saved models/{task}_classifier.pt  best val macro-F1 {best:.3f}  T={temperature:.2f}")


def dice_loss(logits, target):
    p = torch.sigmoid(logits)
    inter = (p * target).sum((1, 2, 3))
    return 1 - ((2 * inter + 1) / (p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + 1)).mean()


def train_unet(epochs, device):
    tr, va = load("seg_train"), load("seg_val")
    xtr, mtr = tr["x"].astype(np.float32), tr["m"].astype(np.float32)
    xva, mva = va["x"].astype(np.float32), va["m"].astype(np.float32)
    model = make_unet(pretrained=True).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    rng = np.random.default_rng(SEED)
    bs, best, best_state, history = 16, -1, None, []
    for ep in range(epochs):
        model.train()
        t0 = time.time()
        idx = rng.permutation(len(xtr))
        for i in range(0, len(idx), bs):
            b = idx[i:i + bs]
            xb = torch.from_numpy(xtr[b][:, None]).to(device)
            mb = torch.from_numpy(mtr[b][:, None]).to(device)
            if rng.random() < 0.5:  # brightness / contrast jitter
                xb = (xb * rng.uniform(0.85, 1.15) + rng.uniform(-0.06, 0.06)).clamp(0, 1)
            out = model(xb)
            loss = F.binary_cross_entropy_with_logits(out, mb) + dice_loss(out, mb)
            opt.zero_grad()
            loss.backward()
            opt.step()
        model.eval()
        dices = []
        with torch.no_grad():
            for i in range(0, len(xva), 32):
                p = (torch.sigmoid(model(torch.from_numpy(xva[i:i + 32][:, None]).to(device))) > 0.5).float().cpu().numpy()[:, 0]
                g = mva[i:i + 32]
                inter = (p * g).sum((1, 2))
                dices += list((2 * inter + 1e-6) / (p.sum((1, 2)) + g.sum((1, 2)) + 1e-6))
        d = float(np.mean(dices))
        history.append({"epoch": ep + 1, "val_dice": d, "seconds": round(time.time() - t0, 1)})
        print(history[-1], flush=True)
        if d > best:
            best, best_state = d, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    torch.save({"state_dict": best_state, "encoder": "mobilenet_v2", "input_size": imaging.SEG_SIZE}, MODELS / "lung_unet.pt")
    (EXPERIMENTS / "unet_training.json").write_text(json.dumps({"history": history, "best_val_dice": best}, indent=2))
    print(f"saved models/lung_unet.pt  best val Dice {best:.3f}")


@torch.no_grad()
def embed(model, x, device):
    e = predict_logits(model, x, device)
    return F.normalize(e, dim=1).numpy()


def train_modality(device):
    """Reference bank of ImageNet embeddings for X-ray and CT; the threshold is set on held-out images."""
    model = make_backbone(pretrained=True).to(device)
    xr = load("xray_train")["x"]
    ct = load("ct_train")["x"]
    bank = np.concatenate([embed(model, xr, device), embed(model, ct, device)])
    labels = np.array([0] * len(xr) + [1] * len(ct))
    xv, cv = load("xray_val")["x"], load("ct_val")["x"]
    sims = []
    for x in (xv, cv):
        s = embed(model, x, device) @ bank.T
        sims += list(np.sort(s, axis=1)[:, -5:].mean(1))
    # margin below the weakest held-out chest images; natural photos score far lower (~0.2-0.35)
    threshold = float(np.clip(np.percentile(sims, 0.5) - 0.1, 0.4, 0.6))
    torch.save(model.state_dict(), MODELS / "modality_backbone.pt")
    np.savez_compressed(MODELS / "modality_bank.npz", bank=bank.astype(np.float16), labels=labels, threshold=threshold)
    print(f"modality bank {bank.shape}, threshold {threshold:.3f}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("task", choices=["xray", "ct", "unet", "modality"])
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    seed_all()
    EXPERIMENTS.mkdir(exist_ok=True)
    if a.task in ("xray", "ct"):
        train_classifier(a.task, a.epochs or (5 if a.task == "xray" else 10), a.device)
    elif a.task == "unet":
        train_unet(a.epochs or 4, a.device)
    else:
        train_modality(a.device)


In [ ]:
%%writefile work/ml/eval.py
"""Evaluate every model on held-out test data -> experiments/eval/metrics.json + confusion matrix plots.

python ml/eval.py
Uses the cached test arrays from ml/prepare_data.py. If they are missing (fresh clone without the
full datasets), it falls back to the committed data: the CT test split and data/sample/.
"""
import datetime
import json
import os
import pathlib

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import torch  # noqa: E402
import torch.nn.functional as F  # noqa: E402
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support  # noqa: E402

from common import CACHE, CT_CLASSES, EXPERIMENTS, MODELS, ROOT, XRAY_CLASSES, imaging  # noqa: E402
from app.services.nets import make_backbone, make_classifier, make_unet  # noqa: E402

OUT = EXPERIMENTS / "eval"
torch.set_num_threads(os.cpu_count() or 4)


def load_cache(name):
    p = CACHE / f"{name}.npz"
    return np.load(p) if p.exists() else None


def load_folder(folder, classes, dir_to_label):
    xs, ys = [], []
    for d in sorted(os.listdir(folder)):
        label = dir_to_label(d)
        for f in sorted(os.listdir(os.path.join(folder, d))):
            with open(os.path.join(folder, d, f), "rb") as fh:
                gray, _, _ = imaging.load_image(fh.read(), f)
            xs.append(imaging.preprocess(gray)["enhanced"])
            ys.append(label)
    return {"x": np.stack(xs), "y": np.array(ys)}


@torch.no_grad()
def logits_for(model, x, bs=64):
    out = []
    for i in range(0, len(x), bs):
        out.append(model(torch.from_numpy(np.stack([imaging.to_tensor_array(im) for im in x[i:i + bs]]))))
    return torch.cat(out)


def plot_cm(cm, classes, title, path):
    fig, ax = plt.subplots(figsize=(6.2, 5.2))
    norm = cm / cm.sum(1, keepdims=True).clip(min=1)
    ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(classes)), classes, rotation=35, ha="right", fontsize=8)
    ax.set_yticks(range(len(classes)), classes, fontsize=8)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=9, color="white" if norm[i, j] > 0.5 else "black")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


def eval_classifier(task, classes, data, source):
    ckpt = torch.load(MODELS / f"{task}_classifier.pt", map_location="cpu")
    model = make_classifier(len(classes))
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    logits = logits_for(model, data["x"])
    probs = torch.softmax(logits / ckpt["temperature"], 1).numpy()
    y, pred = data["y"], probs.argmax(1)
    p, r, f1, s = precision_recall_fscore_support(y, pred, labels=range(len(classes)), zero_division=0)
    cm = confusion_matrix(y, pred, labels=range(len(classes)))
    plot_cm(cm, classes, f"{task.upper()} classifier - test set", OUT / f"confusion_{task}.png")
    conf = probs.max(1)
    bins = np.linspace(0, 1, 11)
    ece = sum(abs((pred[m] == y[m]).mean() - conf[m].mean()) * m.mean()
              for lo, hi in zip(bins[:-1], bins[1:]) if (m := (conf > lo) & (conf <= hi)).any())
    return {
        "source": source, "test_images": int(len(y)), "accuracy": float(accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, average="macro")), "expected_calibration_error": float(ece),
        "per_class": {c: {"sensitivity": float(r[i]), "precision": float(p[i]), "f1": float(f1[i]), "support": int(s[i])} for i, c in enumerate(classes)},
        "confusion_matrix": cm.tolist(), "classes": classes,
    }


@torch.no_grad()
def eval_segmentation():
    data = load_cache("seg_test")
    source = "COVID-19 Radiography Database masks (cached test split)"
    if data is None:  # committed sample
        base = ROOT / "data" / "sample" / "lung_masks"
        import cv2

        xs, ms = [], []
        for f in sorted(os.listdir(base / "images")):
            with open(base / "images" / f, "rb") as fh:
                gray, _, _ = imaging.load_image(fh.read(), f)
            xs.append(imaging.to_seg_array(gray)[0])
            m = cv2.imread(str(base / "masks" / f), cv2.IMREAD_GRAYSCALE)
            ms.append(cv2.resize(m, (imaging.SEG_SIZE, imaging.SEG_SIZE), interpolation=cv2.INTER_NEAREST) > 127)
        data, source = {"x": np.stack(xs), "m": np.stack(ms)}, "data/sample/lung_masks"
    ckpt = torch.load(MODELS / "lung_unet.pt", map_location="cpu")
    model = make_unet()
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    x, m = data["x"].astype(np.float32), data["m"].astype(bool)
    dice, iou = [], []
    for i in range(0, len(x), 32):
        p = (torch.sigmoid(model(torch.from_numpy(x[i:i + 32][:, None]))) > 0.5).numpy()[:, 0]
        g = m[i:i + 32]
        inter = (p & g).sum((1, 2))
        union = (p | g).sum((1, 2))
        dice += list(2 * inter / (p.sum((1, 2)) + g.sum((1, 2)) + 1e-6))
        iou += list(inter / (union + 1e-6))
    return {"source": source, "test_images": int(len(x)), "dice": float(np.mean(dice)), "iou": float(np.mean(iou)),
            "dice_std": float(np.std(dice))}


@torch.no_grad()
def eval_modality(xray, ct):
    """Route chest images to the right model and reject non-chest images (same logic as the API)."""
    from sklearn.datasets import load_sample_images

    bank = np.load(MODELS / "modality_bank.npz")
    emb_bank = torch.from_numpy(bank["bank"].astype(np.float32))
    labels, threshold = bank["labels"], float(bank["threshold"])
    model = make_backbone()
    model.load_state_dict(torch.load(MODELS / "modality_backbone.pt", map_location="cpu"))
    model.eval()

    def decide(x):
        e = F.normalize(logits_for(model, x), dim=1)
        sims = (e @ emb_bank.T).numpy()
        sx = np.sort(sims[:, labels == 0], 1)[:, -5:].mean(1)
        sc = np.sort(sims[:, labels == 1], 1)[:, -5:].mean(1)
        route = (sc > sx).astype(int)
        accepted = np.maximum(sx, sc) >= threshold
        return route, accepted

    # non-chest set: crops / rotations / grey versions of scikit-learn's two bundled photographs
    rng = np.random.default_rng(0)
    others = []
    for img in load_sample_images().images:
        grey = np.array(np.dot(img[..., :3], [0.299, 0.587, 0.114]), dtype=np.uint8)
        for _ in range(20):
            h, w = grey.shape
            s = int(rng.integers(160, min(h, w)))
            y0, x0 = int(rng.integers(0, h - s)), int(rng.integers(0, w - s))
            crop = np.rot90(grey[y0:y0 + s, x0:x0 + s], int(rng.integers(0, 4)))
            others.append(imaging.preprocess(np.ascontiguousarray(crop))["enhanced"])
    others = np.stack(others)
    rx, ax = decide(xray["x"][:300])
    rc, ac = decide(ct["x"])
    _, ao = decide(others)
    routed = np.concatenate([rx == 0, rc == 1])
    accepted = np.concatenate([ax, ac])
    return {"test_images": int(len(routed) + len(others)), "routing_accuracy": float(routed.mean()),
            "chest_accept_rate": float(accepted.mean()), "other_reject_rate": float((~ao).mean()),
            "other_images": int(len(others)), "threshold": threshold,
            "note": "Non-chest set = 40 grey crops of two bundled photographs; colour images are also rejected by a saturation check before this step."}


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    xray = load_cache("xray_test")
    xray_src = "COVID-19 Radiography DB + TB Chest X-ray DB (held-out test split)"
    if xray is None:
        xray = load_folder(ROOT / "data" / "sample" / "xray", XRAY_CLASSES,
                           lambda d: [c.replace(" ", "_").replace("-", "") for c in XRAY_CLASSES].index(d))
        xray_src = "data/sample/xray (small committed sample)"
    ct = load_cache("ct_test")
    ct_src = "Chest CT-Scan Images, published test split (duplicates of train/valid removed)"
    if ct is None:
        from prepare_data import CT_DIRS

        ct = load_folder(ROOT / "data" / "chest_ct" / "test", CT_CLASSES, lambda d: next(v for k, v in CT_DIRS.items() if d.startswith(k)))
        ct_src = "data/chest_ct/test (full published test split)"
    metrics = {
        "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "segmentation": eval_segmentation(),
        "xray": eval_classifier("xray", XRAY_CLASSES, xray, xray_src),
        "ct": eval_classifier("ct", CT_CLASSES, ct, ct_src),
        "modality": eval_modality(xray, ct),
        "notes": [
            "All models were trained with a small CPU configuration (balanced subsets, 224 px, a few epochs); see experiments/*_training.json.",
            "Confidence is temperature-scaled on the validation split; expected calibration error is reported per classifier.",
            "The CT dataset contains many near-duplicate slices; exact duplicates across splits were removed before evaluation.",
        ],
    }
    (OUT / "metrics.json").write_text(json.dumps(metrics, indent=2))
    s, x, c, m = metrics["segmentation"], metrics["xray"], metrics["ct"], metrics["modality"]
    print(f"Segmentation  Dice {s['dice']:.3f}  IoU {s['iou']:.3f}  (n={s['test_images']})")
    print(f"X-ray         acc {x['accuracy']:.3f}  macro-F1 {x['macro_f1']:.3f}  (n={x['test_images']})")
    print(f"CT            acc {c['accuracy']:.3f}  macro-F1 {c['macro_f1']:.3f}  (n={c['test_images']})")
    print(f"Modality      routing {m['routing_accuracy']:.3f}  chest accepted {m['chest_accept_rate']:.3f}  other rejected {m['other_reject_rate']:.3f}")
    for task in ("xray", "ct"):
        print(task, {k: round(v["sensitivity"], 3) for k, v in metrics[task]["per_class"].items()})


if __name__ == "__main__":
    main()


Larger GPU config: raise the per-class counts in prepare_data.py and the epochs below if you have time.

In [ ]:
%cd work/ml
!python prepare_data.py

In [ ]:
!python train.py modality && python train.py xray --epochs 12 && python train.py ct --epochs 25 && python train.py unet --epochs 10

In [ ]:
!python eval.py

In [ ]:
import shutil
for d in ('models', 'experiments'):
    shutil.copytree(f'/kaggle/working/work/{d}', f'/kaggle/working/{d}', dirs_exist_ok=True)
print(os.listdir('/kaggle/working/models'))